# 带颜色约束的车辆排序问题

**类别：** 调度

来源：[https://www.hexaly.com/templates/car-sequencing-problem-with-colors](https://www.hexaly.com/templates/car-sequencing-problem-with-colors)


## 问题描述

**带涂装车间批次约束的车辆排序问题** 涉及一组汽车的生产调度。这些汽车并不完全相同,基本车型有不同的可选配置。装配线上设有不同的工位以安装各种选装件(空调、变速箱、颜色等)。

该问题最初由汽车制造商雷诺提交给[法国运筹学与决策支持协会(ROADEF) 2005 年挑战赛](https://roadef.org/challenge/2005/en/)。

	

### 学习要点

- 使用 [list decision variable](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html) 表示车辆序列
- [按字典序优化多个目标](https://www.hexaly.com/docs/last/modelingprinciples/multiobjectiveresolution.html)
- 区分[结构性约束与首要目标](https://www.hexaly.com/docs/last/modelingprinciples/modelingprinciples.html#distinguish-constraints-from-first-priority-objectives)
- 使用[非线性算子](https://www.hexaly.com/docs/last/mathematicaloperators/operatorsreference.html#table-of-available-operators-and-functions) 来计算违反数


## 数据

数据文件的格式如下:

- 第 1 行:车辆数量、选项数量、类别数量、最大涂装批次大小、目标顺序、起始位置。
- 对于每个选项:在该块中具有该选项的最大车辆数、该块的大小、该选项是否为高优先级。
- 对于每个类别:颜色、该类别的车辆数量、对于每个选项,此类别是否需要该选项(1 或 0)。
- 对于起始位置之前的每个位置:最初计划生产的类别

更多细节,请参阅[挑战赛网站](https://www.roadef.org/challenge/2005/en/sujet.php)。


## 建模方法

带涂装车间批次约束的车辆排序问题的 Hexaly 模型使用 [list decision variable](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html)表示车辆序列。列表中的第 i 个元素对应于第 i 个生产的车辆的索引。由于每辆车必须恰好生产一次,因此我们对该列表变量施加排列约束。

由该序列,我们可以针对每个选项与生产线上的每个位置,计算在该位置开始的窗口中具有该选项的车辆数。据此可以推导出每个选项与每个窗口的违反数。类似地,我们使用 **neq** 与 **or** 算子对涂装车间批次大小施加约束。

尽管该问题是一个纯可行性问题,我们仍选择添加目标,以最小化所有选项与所有窗口的容量违反数之和。事实上,没有容量违反更像是一种"业务"约束,而非结构性约束。如果存在少量违反,装配线仍可继续生产,只需暂时调整生产节奏即可。

我们定义三个目标:

- 最小化高优先级选项的窗口容量违反数
- 最小化低优先级选项的窗口容量违反数
- 最小化颜色变更次数

这三个目标按[字典序进行优化](https://www.hexaly.com/docs/last/modelingprinciples/multiobjectiveresolution.html):声明顺序即定义了其重要性顺序。


## 结果

在**带涂装车间批次约束的车辆排序问题**上,Hexaly Optimizer 在最多 400 辆车的算例上,60 秒运行时间内即可达到近似最优解。在与比赛中使用的 10 分钟求解时间进行比较时,Hexaly 在该问题上的求解规模显著优于传统通用优化求解器。

我们的[专项基准测试页面](https://www.hexaly.com/benchmark/hexaly-vs-gurobi-on-the-car-sequencing-problem-with-paint-shop-batching-constraints)展示了 Hexaly Optimizer 如何在该具有挑战性的问题上优于 Gurobi 等传统通用优化求解器。

[查看该基准](https://www.hexaly.com/benchmark/hexaly-vs-gurobi-on-the-car-sequencing-problem-with-paint-shop-batching-constraints)


## Python 实现


In [ ]:
# Copyright (c) Hexaly. Permission is hereby granted to use, copy,
# and modify this code for applications developed with Hexaly.
import hexaly.optimizer
import sys

COLOR_HIGH_LOW = 0
HIGH_LOW_COLOR = 1
HIGH_COLOR_LOW = 2
COLOR_HIGH = 3
HIGH_COLOR = 4

def read_integers(filename):
    with open(filename) as f:
        return [int(elem) for elem in f.read().split()]

#
# Read instance data
#
def read_instance(instance_file):
    file_it = iter(read_integers(instance_file))
    nb_positions = next(file_it)
    nb_options = next(file_it)
    nb_classes = next(file_it)
    paint_batch_limit = next(file_it)
    objective_order = next(file_it)
    start_position = next(file_it)

    max_cars_per_window = []
    window_size = []
    is_priority_option = []
    has_low_priority_options = False

    for o in range(nb_options):
        max_cars_per_window.append(next(file_it))
        window_size.append(next(file_it))
        is_prio = next(file_it) == 1
        is_priority_option.append(is_prio)
        if not is_prio:
            has_low_priority_options = True

    if not has_low_priority_options:
        if objective_order == COLOR_HIGH_LOW:
            objective_order = COLOR_HIGH
        elif objective_order == HIGH_COLOR_LOW:
            objective_order = HIGH_COLOR
        elif objective_order == HIGH_LOW_COLOR:
            objective_order = HIGH_COLOR

    color_class = []
    nb_cars = []
    options_data = []

    for c in range(nb_classes):
        color_class.append(next(file_it))
        nb_cars.append(next(file_it))
        options_data.append([next(file_it) == 1 for i in range(nb_options)])

    initial_sequence = [next(file_it) for p in range(nb_positions)]

    return nb_positions, nb_options, paint_batch_limit, objective_order, start_position, \
        max_cars_per_window, window_size, is_priority_option, has_low_priority_options, \
        color_class, options_data, initial_sequence

def main(instance_file, output_file, time_limit):
    nb_positions, nb_options, paint_batch_limit, objective_order, start_position, \
        max_cars_per_window, window_size, is_priority_option, has_low_priority_options, \
        color_class, options_data, initial_sequence = read_instance(instance_file)

    with hexaly.optimizer.HexalyOptimizer() as optimizer:
        #
        # Declare the optimization model
        #
        model = optimizer.model

        # sequence[i] = j if class initially planned on position j is produced on position i
        sequence = model.list(nb_positions)

        # sequence is a permutation of the initial production plan, all indexes must appear
        # exactly once
        model.constraint(model.partition(sequence))

        # Past classes (before startPosition) can not move
        [model.constraint(sequence[p] == p) for p in range(start_position)]

        # Create Hexaly arrays to be able to access them with "at" operators
        initials = model.array(initial_sequence)
        colors = model.array(color_class)
        options = model.array(options_data)

        # Number of cars with option o in each window
        nb_cars_windows = [None] * nb_options
        for o in range(nb_options):
            nb_cars_windows[o] = [None] * nb_positions
            for j in range(start_position - window_size[o] + 1, nb_positions):
                nb_cars_windows[o][j] = model.sum()
                for k in range(window_size[o]):
                    if j + k >= 0 and j + k < nb_positions:
                        class_at_position = initials[sequence[j + k]]
                        nb_cars_windows[o][j].add_operand(model.at(
                            options,
                            class_at_position,
                            o))

        # Number of violations of option o capacity in each window
        objective_high_priority = model.sum()
        if has_low_priority_options:
            objective_low_priority = model.sum()

        for o in range(nb_options):
            nb_violations_windows = model.sum(
                model.max(
                    nb_cars_windows[o][p] - max_cars_per_window[o], 0)
                    for p in range(start_position - window_size[o] + 1, nb_positions))
            if is_priority_option[o]:
                objective_high_priority.add_operand(nb_violations_windows)
            else:
                objective_low_priority.add_operand(nb_violations_windows)

        # Color change between position p and position p + 1
        color_change = [None] * (nb_positions - 1)
        objective_color = model.sum()
        for p in range(start_position - 1, nb_positions - 1):
            current_class = initials[sequence[p]]
            next_class = initials[sequence[p + 1]]
            color_change[p] = colors[current_class] != colors[next_class]
            objective_color.add_operand(color_change[p])

        # Paint limit constraints: at least one change every paintBatchLimit positions
        for p in range(start_position, nb_positions - paint_batch_limit - 1):
            node_or = model.or_(color_change[p + p2] for p2 in range(paint_batch_limit))
            model.constraint(node_or)

        # Declare the objectives in the correct order
        if objective_order == COLOR_HIGH_LOW:
            model.minimize(objective_color)
            model.minimize(objective_high_priority)
            model.minimize(objective_low_priority)
        elif objective_order == HIGH_COLOR_LOW:
            model.minimize(objective_high_priority)
            model.minimize(objective_color)
            model.minimize(objective_low_priority)
        elif objective_order == HIGH_LOW_COLOR:
            model.minimize(objective_high_priority)
            model.minimize(objective_low_priority)
            model.minimize(objective_color)
        elif objective_order == COLOR_HIGH:
            model.minimize(objective_color)
            model.minimize(objective_high_priority)
        elif objective_order == HIGH_COLOR:
            model.minimize(objective_high_priority)
            model.minimize(objective_color)

        model.close()


        # Set the initial solution
        sequence.get_value().clear()
        for p in range(nb_positions):
            sequence.get_value().add(p)

        # Parameterize the optimizer
        optimizer.param.time_limit = time_limit

        optimizer.solve()

        #
        # Write the solution in a file with the following format:
        # - 1st line: value of the objectives;
        # - 2nd line: for each position p, index of class at positions p.
        #
        if output_file is not None:
            with open(output_file, 'w') as f:
                f.write("%d " % objective_color.value)
                f.write("%d " % objective_high_priority.value)
                f.write("%d\n" % objective_low_priority.value)
                for p in range(nb_positions):
                    f.write("%d " % sequence.value[p])

                f.write("\n")

if __name__ == '__main__':
    if len(sys.argv) < 2:
        print("Usage: python car_sequencing_color.py instance_file [output_file] [time_limit]")
        sys.exit(1)

    instance_file = sys.argv[1]
    output_file = sys.argv[2] if len(sys.argv) >= 3 else None
    time_limit = int(sys.argv[3]) if len(sys.argv) >= 4 else 60
    main(instance_file, output_file, time_limit)
